# Chapter 09 답안 양식. 회귀 분석으로 숫자 예측하기

> 이 내용을 `chapter09.ipynb`의 Markdown 셀로 작성합니다.

## 제출 정보
- 이름: 황수아
- GitHub ID: totoriri605-oss
- 작성일: 2026.10.01
- 최종 Notebook URL: https://github.com/totoriri605-oss/llm-data-analysis-course/blob/main/chapter09/chapter09.ipynb

> 참고: 내 로컬에는 강의용 `customers.csv`(고객 정보)랑 주문상세 파일이 없어서(`customers.csv`가 주문 파일이랑 똑같은 내용이었음)
> 고객 정보와 주문상세는 `scripts/preprocess_data.py`에서 seed 고정으로 **가상 생성**한 데이터를 썼다. 주문 300건(`order.csv`)만 실제 파일이다.
> 그래서 아래 숫자는 방법을 따라해 본 결과이고, 업무 결론으로 쓰기에는 한계가 있다.

In [1]:
import subprocess
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
report_dir = project_root / "reports"
print("프로젝트 루트:", project_root)

프로젝트 루트: C:\dev\llm-data-analysis-course


## 1. 예측 문제 정의
- Target: `order_total` (주문별 `line_total` 합계)
- 예측 단위: 주문 1건
- 예측 시점: 주문 날짜, 결제수단, 고객 정보(나이/성별/지역)는 알지만 주문상세(수량/단가/금액)는 아직 모델 입력에 없는 시점
- 예측 시점에 알 수 있는 정보:
  `order_month`, `order_dayofweek`, `age`, `payment_method`, `gender`, `city`
- 예측 시점에 알 수 없는 정보:
  `order_total`, `line_total`, `quantity`, `unit_price`, `item_count`, `total_quantity`, `avg_unit_price`, `order_status`, `order_id`, `customer_id`, `product_id`

### 나의 해석과 판단
주문상세(수량, 단가)까지 알면 주문금액은 곱해서 더하면 끝이라 예측이 아니라 계산이 된다고 책에 나와 있다.
그래서 "주문이 들어왔다는 것과 고객 정보만 아는 시점"에서 이 주문이 대략 얼마짜리일지 맞혀 보는 문제로 정했다.
이 시점이어야 입력으로 쓸 수 있는 값과 못 쓰는 값을 나누는 연습(누수 막기)이 의미가 있다.
다만 이건 교육용으로 만든 가정이고, 실제 업무라면 이 시점에 어떤 정보가 진짜 있는지부터 확인해야 할 것 같다.

---

## 2. Feature Leakage Audit

| feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유 |
| --- | --- | --- | --- | --- | --- |
| order_month | O | X | X | O | 주문일에서 만든 값, 예측 시점에 알 수 있다고 가정 |
| order_dayofweek | O | X | X | O | 주문일에서 만든 값, 예측 시점에 알 수 있다고 가정 |
| age | O | X | X | O | 고객 비식별 특성 |
| payment_method | O | X | X | O | 주문 메타데이터 |
| gender | O | X | X | O | 고객 비식별 특성 |
| city | O | X | X | O | 고객 비식별 특성 |
| order_total | X | O | X | X | 목표값 자체 |
| line_total | X | O | X | X | 목표값 계산 재료 |
| quantity | X | O | X | X | 목표값 계산 재료 |
| unit_price | X | O | X | X | 목표값 계산 재료 |
| item_count | X | O | X | X | 주문상세 집계라 목표값과 연결됨(대리 변수) |
| total_quantity | X | O | X | X | 주문상세 집계, 목표값을 거의 복원할 수 있는 대리 변수 |
| avg_unit_price | X | O | X | X | 주문상세 집계, 목표값을 거의 복원할 수 있는 대리 변수 |
| order_status | X | X | O | X | 주문 처리 후 정해지는 사후정보일 수 있음 |
| order_id | X | X | O | X | 식별자 (숫자여도 크기 의미 없음) |
| customer_id | X | X | O | X | 식별자 (고객 암기 위험) |
| product_id | X | X | O | X | 식별자 |

- 금지 feature overlap 개수: **0개** (`ch09_regression_validation.csv` 기준)
- 가장 위험하다고 판단한 feature: `total_quantity`, `avg_unit_price` (주문상세 집계)
- 이유: 직접적인 정답 컬럼은 아니라서 방심하기 쉬운데, 실제로 이 둘을 입력에 넣어 봤더니(`notebooks/ch09/ch09_assignments.ipynb` 과제 3)
  Random Forest 기준 test R²가 -0.05에서 0.88로 올라갔다. 점수는 좋아 보이지만 예측 시점에는 모르는 값이라 쓸 수 없는 모델이 된다.

![Leakage 점검](images/step02_leakage.png)

---

In [2]:
audit = pd.read_csv(report_dir / "ch09_regression_feature_audit.csv")
display(audit)
print("선택된 입력:", audit[audit["selected"]]["column"].tolist())

,column,selected,role,reason
0,order_month,True,allowed_feature,"주문일에서 만든 값, 예측 시점에 알 수 있음"
1,order_dayofweek,True,allowed_feature,"주문일에서 만든 값, 예측 시점에 알 수 있음"
2,age,True,allowed_feature,고객 비식별 특성
3,payment_method,True,allowed_feature,주문 메타데이터
4,gender,True,allowed_feature,고객 비식별 특성
5,city,True,allowed_feature,고객 비식별 특성
6,avg_unit_price,False,forbidden,주문상세 집계 (목표값과 연결)
7,customer_id,False,forbidden,식별자
8,item_count,False,forbidden,주문상세 집계 (목표값과 연결)
9,line_total,False,forbidden,목표값 계산 재료


선택된 입력: ['order_month', 'order_dayofweek', 'age', 'payment_method', 'gender', 'city']


## 3. Target 생성과 관계 검증

### 계산 기준

```text
line_total = quantity * unit_price
order_total = order_id별 line_total 합계
```

- line_total 검증 결과: 주문상세 746행 중 746행이 `quantity × unit_price`와 일치 (불일치 0행)
- orders ↔ target 관계 검증 결과: `one_to_one` 통과 (주문 300건, 목표값 300건, 한쪽에만 있는 주문 없음)
- orders → customers 관계 검증 결과: `many_to_one` 통과 (고객 한 명이 여러 주문, 병합 전후 행 수 동일)

### 나의 해석과 판단
관계를 확인하지 않고 그냥 merge만 하면, 주문상세가 없는 주문이 조용히 빠지거나 키가 중복돼서 행이 늘어날 수 있다.
행이 늘면 같은 주문이 여러 번 세어져서 order_total이나 평균/오차 같은 숫자가 부풀거나 줄어드는데, 에러가 안 나서 모르고 지나가기 쉽다.
그래서 `validate="one_to_one"`, `validate="many_to_one"`를 쓰고 병합 전후 행 수도 비교하게 해 뒀다.

---

In [3]:
from src.regression import load_regression_source_data, build_regression_dataset

datasets = load_regression_source_data(project_root / "data" / "processed")
items = datasets["order_items"]
mismatch = ((items["quantity"] * items["unit_price"] - items["line_total"]).abs() > 1e-6).sum()
print("주문상세 행 수:", len(items), "/ line_total 불일치:", mismatch)

model_data = build_regression_dataset(datasets["customers"], datasets["orders"], items)
print("모델링 데이터:", model_data.shape, "/ 주문ID 중복:", model_data["order_id"].duplicated().sum())

주문상세 행 수: 746 / line_total 불일치: 0
모델링 데이터: (300, 11) / 주문ID 중복: 0


## 4. Train / Final Test 분할

- Train 시작일: 2025-07-09
- Train 종료일: 2026-04-08
- Train 행 수: 242 (80.7%)
- Final Test 시작일: 2026-04-10
- Final Test 종료일: 2026-07-08
- Final Test 행 수: 58 (19.3%)
- 같은 달력 날짜 중복 여부: 없음 (겹치는 날짜 0개)

### 분할 판단
실제로는 과거 주문으로 학습해서 앞으로 들어올 주문을 예측하니까, 과거를 train, 미래를 test로 나눴다.
무작위로 섞으면 미래 주문이 train에 들어가서 실제보다 쉬운 평가가 된다. (과제 4에서 무작위 분할을 해 보니 같은 날짜 27개가 양쪽에 겹쳤다.)
같은 날짜는 한쪽에만 넣어서 정확히 80:20은 아니고 80.7:19.3가 됐는데, 비율보다 시간 경계가 엄격한 게 더 중요하다고 해서 그대로 뒀다.

![시간 분할](images/step04_split.png)

---

In [4]:
split_summary = pd.read_csv(report_dir / "ch09_regression_split_summary.csv")
display(split_summary)
print("strict_train_before_test:", split_summary.loc[0, "end_date"] < split_summary.loc[1, "start_date"])

,split,rows,ratio_pct,start_date,end_date
0,train,242,80.7,2025-07-09,2026-04-08
1,test,58,19.3,2026-04-10,2026-07-08


strict_train_before_test: True


## 5. Train TimeSeriesSplit 후보 비교

| 모델 | CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 | 비고 |
| --- | ---: | ---: | ---: | --- |
| Baseline Mean | 95,350.46 | 14,097.40 | -0.0512 | 단순 평균 기준 (선택 대상 아님) |
| Linear Regression | 115,127.62 | 12,535.45 | -0.5735 | 비베이스라인 후보 |
| Random Forest | 99,490.62 | 11,229.82 | -0.2217 | 선택됨 (비베이스라인 중 CV MAE 평균 최소) |

### Train CV로 선택한 비베이스라인 모델
- Selected Model: **Random Forest**
- 선택 기준: 비베이스라인(Linear, Random Forest) 중 `cv_MAE_mean`이 가장 낮은 모델
- Final Test 결과를 보기 전에 선택했는가: **예** (훈련 기간 TimeSeriesSplit 5개 fold로만 선택하고 고정)

### 나의 해석과 판단
평균 MAE만 보면 기간마다 성능이 들쭉날쭉한 모델도 괜찮아 보일 수 있다. 표준편차는 fold(기간)마다 오차가 얼마나 흔들리는지 알려줘서,
평균이 비슷해도 표준편차가 큰 모델은 어느 기간에 얼마나 틀릴지 예측하기 어렵다. 이번엔 Random Forest가 Linear보다 평균도 낮고 표준편차도 낮아서 선택 이유가 일관됐다.
그런데 Random Forest도 CV 평균 MAE(99,491)가 Baseline(95,350)보다 높았다.
훈련 기간 안에서는 모델이 단순 평균을 못 이겼다는 뜻이라, 이 점은 뒤의 해석에서 같이 고려해야 한다.

---

In [5]:
cv_summary = pd.read_csv(report_dir / "ch09_regression_cv_summary.csv")
display(cv_summary.round(2))

,model,n_splits,cv_MAE_mean,cv_MAE_std,cv_R2_mean
0,Baseline Mean,5,95350.46,14097.40,-0.05
1,Linear Regression,5,115127.62,12535.45,-0.57
2,Random Forest,5,99490.62,11229.82,-0.22


## 6. Final Test: Baseline vs Frozen Model

| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Baseline Mean | baseline | 95,324 | 127,791 | 156,604 | -0.0036 | 0.00% |
| Frozen Selected Model (Random Forest) | selected_by_train_cv | 74,650 | 121,477 | 160,267 | -0.0511 | 4.94% |

![최종 회귀 평가](images/step06_final_metrics.png)

### 결과 관찰
- Baseline Mean의 Test MAE는 약 127,791, Random Forest은 약 121,477으로 MAE만 보면 약 4.9% 낮다.
- Test RMSE는 Baseline 156,604, Random Forest 160,267으로 오히려 약 2.3% 높다.
- Test R²는 -0.0511로 음수다. (Baseline도 -0.0036로 사실상 0)
- Train MAE는 약 74,650인데 Test MAE는 약 121,477으로 약 46,827 차이가 난다.

### 나의 해석과 판단
- Frozen Model이 baseline보다 실제로 개선되었는가? MAE는 조금 낮아졌지만 RMSE와 R²는 오히려 나빠졌고, 훈련 CV에서도 Baseline보다 MAE가 높았다.
  그래서 "실제로 개선됐다"고 말하기는 어렵다. 4.9%는 우연일 수도 있다.
- MAE와 RMSE 차이에서 무엇을 볼 수 있는가? RMSE는 큰 오차에 더 민감한데 RMSE만 나빠졌다는 건, 평균적인 오차는 조금 줄었어도 큰 오차(큰 주문을 낮게 맞히는 경우)가 여전히 크다는 뜻으로 보인다.
- R²는 어떻게 해석할 수 있는가? 음수는 테스트 목표값의 평균으로 예측하는 것보다 설명을 못 했다는 뜻이다. 코드 오류라고 단정할 수는 없고, 입력 변수에 금액 신호가 약하다는 쪽에 가깝다.
- Train MAE가 Test MAE보다 꽤 낮은 건 Random Forest가 train에 과적합했을 가능성이 있다는 신호로 본다.

### 업무·분석적 의미
Test MAE 121,477원은 test 주문의 평균 실제 금액(157,552원)의 약 77% 수준이다.
즉 예측이 평균적으로 주문금액의 거의 8할만큼 빗나간다는 뜻이라, 이 모델로 개별 주문 금액을 알려주는 용도로 쓰기에는 오차가 너무 크다.
(가상 데이터라서 실제 업무 수치로 받아들이면 안 된다.)

---

In [6]:
model_comparison = pd.read_csv(report_dir / "ch09_regression_model_comparison.csv")
display(model_comparison.round(2))

,model,selection_role,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,baseline,95324.12,127790.68,156604.49,-0.00,0.00
1,Random Forest,selected_by_train_cv,74649.53,121476.77,160266.70,-0.05,4.94


## 7. 대표 오차 사례

### 사례 1
- 실제값: 730,000원 (주문일 2026-04-13)
- 예측값: 약 95,883원
- residual: +634,117원 (실제값 - 예측값, 양수라서 과소예측)
- absolute error: 634,117원
- 관찰: test 구간에서 절대오차가 가장 컸고, 실제 금액이 test 평균보다 훨씬 큰 주문이다. 실제값이 예측값의 약 7.6배다.
- 가능한 이유 후보: 현재 입력(월, 요일, 나이, 결제수단, 성별, 지역)으로는 주문이 큰지 작은지 구분할 정보가 거의 없어서, 큰 주문도 평균 근처로 예측했을 가능성이 있다. 주문상세에 비싼 상품이나 수량이 많이 들어갔을 수도 있지만 이 입력에는 없어서 확인할 수 없다.
- 추가 확인: 이 주문의 주문상세(수량/단가)가 다른 주문과 얼마나 다른지, 데이터 입력 오류는 아닌지 확인.


### 사례 2
- 실제값: 519,000원 (주문일 2026-07-05)
- 예측값: 약 137,115원
- residual: +381,885원 (실제값 - 예측값, 양수라서 과소예측)
- absolute error: 381,885원
- 관찰: 절대오차 2위이고, 사례 1과 마찬가지로 큰 주문을 낮게 예측한 과소예측이다. 실제값이 예측값의 약 3.8배다.
- 가능한 이유 후보: 사례 1과 같은 이유 후보로 보이지만 같은 원인인지는 알 수 없다. 특정 기간에 큰 주문이 몰렸을 수도 있다.
- 추가 확인: 상위 오차 주문들이 특정 날짜/결제수단에 몰려 있는지, 예측 시점 이전에 알 수 있는 추가 변수(과거 구매 이력 등)가 있는지 확인.


> 원인을 데이터가 직접 보여 주지 않는다면 사실처럼 단정하지 않습니다.
> (위 이유 후보는 가설이고, 이 데이터로는 확정할 수 없다.)

---

In [7]:
prediction_result = pd.read_csv(report_dir / "ch09_regression_predictions_internal.csv")
display(prediction_result.head(2).round(0))
print("과소예측 비율 (상위 10건):", (prediction_result.head(10)["residual"] > 0).mean())

,order_id,order_date,actual_order_total,predicted_order_total,residual,abs_error,model
0,84,2026-04-13,730000.0,95883.0,634117.0,634117.0,Random Forest
1,264,2026-07-05,519000.0,137115.0,381885.0,381885.0,Random Forest


과소예측 비율 (상위 10건): 1.0


## 8. Validation Evidence

`reports/ch09_regression_validation.csv` 기준으로 작성하세요.

| check | value | status |
| --- | --- | --- |
| forbidden_feature_overlap | 0 | PASS |
| strict_train_before_test | True | PASS |
| selected_model_exists_in_train_cv | True | PASS |
| final_test_contains_baseline | True | PASS |
| final_test_contains_frozen_selected_model | True | PASS |
| test_rows_for_r2 | 58 | PASS |

(CSV에는 추가로 `no_shared_calendar_dates`, `selected_model_is_not_baseline`, `final_test_only_two_models` 3개도 있고 모두 PASS)

### FAIL 항목
- 없음 / 있음: **없음** (총 9개 check 중 FAIL 0개)
- 있다면 원인과 수정 내용: 해당 없음

---

In [8]:
validation = pd.read_csv(report_dir / "ch09_regression_validation.csv")
display(validation)
print("FAIL 개수:", (validation["status"] != "PASS").sum())

,check,value,status,passed,detail
0,forbidden_feature_overlap,0,PASS,True,[]
1,strict_train_before_test,True,PASS,True,2026-04-08 < 2026-04-10
2,no_shared_calendar_dates,0,PASS,True,겹치는 날짜 수
3,selected_model_exists_in_train_cv,True,PASS,True,Random Forest
4,selected_model_is_not_baseline,True,PASS,True,Random Forest
5,final_test_contains_baseline,True,PASS,True,NaN
6,final_test_contains_frozen_selected_model,True,PASS,True,Random Forest
7,final_test_only_two_models,2,PASS,True,"['Baseline Mean', 'Random Forest']"
8,test_rows_for_r2,58,PASS,True,R² 계산에는 test 행이 2개 이상 필요


FAIL 개수: 0


## 9. Evidence와 공개 범위

### 생성된 Evidence
- [x] `ch09_regression_split_summary.csv`
- [x] `ch09_regression_feature_audit.csv`
- [x] `ch09_regression_cv_summary.csv`
- [x] `ch09_regression_model_comparison.csv`
- [x] `ch09_regression_validation.csv`
- [x] `ch09_regression_checklist.csv`

### Internal 파일
- [x] `ch09_regression_model_data_internal.csv`
- [x] `ch09_regression_predictions_internal.csv`

(Internal 파일은 로컬에만 있고 `.gitignore`로 GitHub에는 올리지 않았다.)

### 공개 결과
- [x] `ch09_regression_report.md`
- [x] `ch09_actual_vs_predicted.png`
- [x] `ch09_residual_histogram.png`

### 공개 범위 판단
`order_id`는 특정 주문을 가리키는 식별자라서, 예측 결과와 같이 공개하면 실제 주문 한 건 한 건과 연결될 수 있다.
공개 보고서에는 주문 단위 식별이 필요하지 않으니, `order_id`가 들어간 파일은 Internal로 따로 두고 공개용 오차표에서는 뺐다.

---

In [9]:
from src.regression import public_prediction_result

internal = pd.read_csv(report_dir / "ch09_regression_predictions_internal.csv")
public = public_prediction_result(internal)
print("internal에 order_id:", "order_id" in internal.columns)
print("public에 order_id:", "order_id" in public.columns)
for name in ["ch09_regression_model_data_internal.csv", "ch09_regression_predictions_internal.csv"]:
    print(name, (report_dir / name).exists())

internal에 order_id: True
public에 order_id: False
ch09_regression_model_data_internal.csv True
ch09_regression_predictions_internal.csv True


## 10. 최종 사용 판단

- [ ] 현재 수준에서 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 데이터로는 사용 보류

### 판단 근거
1. Test MAE는 Baseline보다 4.9% 낮지만, Test RMSE는 오히려 높고(약 2.3%) Test R²는 -0.051로 음수다.
2. 훈련 기간 TimeSeriesSplit에서도 Random Forest의 CV MAE가 Baseline보다 높아서, Final Test의 소폭 개선을 믿을 근거가 약하다.
3. Validation은 모두 PASS라서 평가 과정(누수 차단, 시간 분리, 선택 고정) 자체는 문제없지만, 결과가 "쓸 만하다"를 뒷받침하지는 못한다.

### 업무적 위험
Test MAE가 test 평균 주문금액의 약 77% 수준이고 큰 주문은 크게 과소예측한다(상위 오차 거의 전부 과소예측).
이 모델로 매출 예측이나 개별 주문 금액 안내를 하면 틀린 판단으로 이어질 수 있다.

### 현재 데이터의 한계
- 입력이 월, 요일, 나이, 결제수단, 성별, 지역뿐이라 주문금액을 설명할 신호가 약하다.
- 고객 정보와 주문상세가 가상 데이터라서 실제 업무 패턴을 반영하지 못한다.
- test가 58행이라 작아서 MAE/R²가 우연히 흔들릴 수 있다.

### 다음 개선 우선순위
1. 예측 시점 이전에 실제로 알 수 있는 변수 추가 (고객 과거 구매 이력, 최근 방문 등). 단, 주문상세 집계 같은 누수 변수는 추가하지 않는다.
2. 진짜 고객/주문상세 데이터로 다시 실행하고, 더 이후 기간을 새로운 검증 기간으로 확보한다.
3. 큰 주문(상위 오차)이 어떤 주문인지 확인해서 데이터 품질 문제인지 현재 입력의 한계인지 구분한다.

---

## 11. 전체 재실행 확인

실행 명령:

```powershell
python scripts/run_regression_analysis.py
```

- 실행 성공 여부: 성공 (종료 코드 0)
- Train CV Selected Model: Random Forest
- Final Validation 전체 PASS 여부: 예 (전체 PASS)
- Notebook 결과와 스크립트 결과의 일관성: 아래 셀에서 스크립트를 다시 실행해 선택 모델과 Final Test 숫자가 이 노트북의 표와 같은지 확인함.

In [10]:
run = subprocess.run(
    [sys.executable, "scripts/run_regression_analysis.py"],
    cwd=project_root, capture_output=True, text=True, encoding="utf-8",
)
print("종료 코드:", run.returncode)
print(run.stdout[-900:])

saved = pd.read_csv(report_dir / "ch09_regression_model_comparison.csv")
print("저장된 선택 모델:", saved[saved["selection_role"] == "selected_by_train_cv"]["model"].iloc[0])
print("Validation 전체 PASS:", (pd.read_csv(report_dir / "ch09_regression_validation.csv")["status"] == "PASS").all())

종료 코드: 0
분할 요약
split  rows  ratio_pct start_date   end_date
train   242       80.7 2025-07-09 2026-04-08
 test    58       19.3 2026-04-10 2026-07-08

훈련 CV 결과
            model  n_splits  cv_MAE_mean  cv_MAE_std  cv_R2_mean
    Baseline Mean         5     95350.46    14097.40       -0.05
Linear Regression         5    115127.62    12535.45       -0.57
    Random Forest         5     99490.62    11229.82       -0.22

선택 모델: Random Forest

Final Test
        model       selection_role  train_MAE  test_MAE  test_RMSE  test_R2  MAE_improvement_vs_baseline_pct
Baseline Mean             baseline   95324.12 127790.68  156604.49    -0.00                             0.00
Random Forest selected_by_train_cv   74649.53 121476.77  160266.70    -0.05                             4.94

Validation 통과: True

저장된 선택 모델: Random Forest
Validation 전체 PASS: True


---

## 최종 체크

- [x] Target과 예측 시점을 정의했습니다.
- [x] Leakage Audit을 수행했습니다.
- [x] Target 계산과 관계 검증을 확인했습니다.
- [x] Train과 Final Test가 시간 순서로 엄격히 분리되었습니다.
- [x] 전처리가 Pipeline 내부에서 학습됩니다.
- [x] Train TimeSeriesSplit으로 후보를 비교했습니다.
- [x] Final Test 전에 Selected Model을 고정했습니다.
- [x] Final Test에서는 Baseline과 Frozen Model만 비교했습니다.
- [x] MAE, RMSE, R²를 함께 해석했습니다.
- [x] 대표 오차 사례를 관찰과 가설로 구분했습니다.
- [x] Validation Evidence가 모두 PASS입니다.
- [x] Internal 결과와 공개 결과를 구분했습니다.
- [x] 낮은 성능도 숨기지 않았습니다.
- [x] 전체 스크립트를 재실행했습니다.
- [ ] 최종 Notebook URL을 제출합니다. (제출은 직접)